In [22]:
import warnings
warnings.filterwarnings('ignore')
import pandas as pd
import pyodbc
from sqlalchemy import create_engine

# Defining connection string
conn_str = 'DRIVER={SQL Server};SERVER=localhost\SQLEXPRESS;DATABASE=ProjectRFM;Trusted_Connection=yes;'

engine = create_engine(f"mssql+pyodbc:///?odbc_connect={conn_str}")

# Read using the engine
query = "SELECT * FROM CleanSalesData"
df = pd.read_sql(query, engine)

print(f"Data successfully pulled! Row count: {len(df)}")
df.head(5)

Data successfully pulled! Row count: 805549


,InvoiceNo,StockCode,Description,CustomerID,InvoiceDate,Quantity,UnitPrice,TotalLineValue
0,502813,22406,MONEY BOX KINGS CHOICE DESIGN,16104,2010-03-28 10:33:00,6,1.25,7.500000
1,502813,22409,MONEY BOX BISCUITS DESIGN,16104,2010-03-28 10:33:00,6,1.25,7.500000
2,502813,22413,METAL SIGN TAKE IT OR LEAVE IT,16104,2010-03-28 10:33:00,6,2.95,17.700000
3,502814,85014A,BLACK/BLUE DOTS RUFFLED UMBRELLA,13767,2010-03-28 10:33:00,12,5.95,71.399998
4,502814,85014B,RED/WHITE DOTS RUFFLED UMBRELLA,13767,2010-03-28 10:33:00,12,5.95,71.399998


In [23]:
# checking for nulls
print(df.isnull().sum()) 

# This line officially removed them
df = df.dropna(subset=['CustomerID'])

InvoiceNo         0
StockCode         0
Description       0
CustomerID        0
InvoiceDate       0
Quantity          0
UnitPrice         0
TotalLineValue    0
dtype: int64


In [24]:
print("--- Dataset Dimensions ---")

# Check dimensions (Rows, Columns)
print(f"Dataset Shape: {df.shape}")

# Check total row count specifically
print(f"Total Rows: {len(df)}")

# Check column names and data types
print(df.info())

--- Dataset Dimensions ---
Dataset Shape: (805549, 8)
Total Rows: 805549
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 805549 entries, 0 to 805548
Data columns (total 8 columns):
 #   Column          Non-Null Count   Dtype         
---  ------          --------------   -----         
 0   InvoiceNo       805549 non-null  object        
 1   StockCode       805549 non-null  object        
 2   Description     805549 non-null  object        
 3   CustomerID      805549 non-null  object        
 4   InvoiceDate     805549 non-null  datetime64[ns]
 5   Quantity        805549 non-null  int64         
 6   UnitPrice       805549 non-null  float64       
 7   TotalLineValue  805549 non-null  float64       
dtypes: datetime64[ns](1), float64(2), int64(1), object(4)
memory usage: 49.2+ MB
None


In [25]:
#Set a "Reference Date" (Usually one day after the last date in the dataset)
import datetime as dt
latest_date = df['InvoiceDate'].max() + dt.timedelta(days=1)

# Group by Customer and calculate Recency, Frequency, and Monetary
rfm = df.groupby('CustomerID').agg({
    'InvoiceDate': lambda x: (latest_date - x.max()).days, # Recency
    'InvoiceNo': 'count',                                 # Frequency
    'TotalLineValue': 'sum'                               # Monetary
})

# Rename columns for clarity
rfm.columns = ['Recency', 'Frequency', 'Monetary']
rfm['Monetary'] = rfm['Monetary'].round(2)

# Filter out any 0 or negative values just in case
rfm = rfm[rfm['Monetary'] > 0]

print("RFM Table Created!")
rfm.head(5)

RFM Table Created!


,Recency,Frequency,Monetary
CustomerID,,,
12346,326,34,77556.46
12347,2,253,5633.32
12348,75,51,2019.40
12349,19,175,4428.69
12350,310,17,334.40


In [26]:
# Calculate scores (1-5)
# For Recency: Lower is better (more recent), so labels are [5, 4, 3, 2, 1]
rfm['R_Score'] = pd.qcut(rfm['Recency'], 5, labels=[5, 4, 3, 2, 1])

# For Frequency and Monetary: Higher is better, so labels are [1, 2, 3, 4, 5]
rfm['F_Score'] = pd.qcut(rfm['Frequency'].rank(method="first"), 5, labels=[1, 2, 3, 4, 5]) 
rfm['M_Score'] = pd.qcut(rfm['Monetary'], 5, labels=[1, 2, 3, 4, 5])

# Combine them into one RFM Score string (like "555" or "111")
rfm['RFM_Score_Group'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str) + rfm['M_Score'].astype(str)

print("Scoring Complete!")
rfm.head(5)

Scoring Complete!


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score_Group
CustomerID,,,,,,,
12346,326,34,77556.46,2,2,5,225
12347,2,253,5633.32,5,5,5,555
12348,75,51,2019.40,3,3,4,334
12349,19,175,4428.69,5,4,5,545
12350,310,17,334.40,2,2,2,222


In [27]:
# Define the rules for each segment using Regex (Regular Expressions)
segs = {
    r'[1-2][1-2]': 'Hibernating',
    r'[1-2][3-4]': 'At Risk',
    r'[1-2]5': 'Can\'t Loose Them',
    r'3[1-2]': 'About To Sleep',
    r'33': 'Need Attention',
    r'[3-4][4-5]': 'Loyal Customers',
    r'41': 'Promising',
    r'51': 'New Customers',
    r'[4-5][2-3]': 'Potential Loyalists',
    r'5[4-5]': 'Champions'
}

# 2. Apply the map to the R and F scores
rfm['Segment'] = rfm['R_Score'].astype(str) + rfm['F_Score'].astype(str)
rfm['Segment'] = rfm['Segment'].replace(segs, regex=True)

print("Segmentation Complete! Here is the final count:")
print(rfm['Segment'].value_counts())
rfm.head(5)

Segmentation Complete! Here is the final count:
Segment
Hibernating            1437
Loyal Customers        1135
Champions               821
At Risk                 802
Potential Loyalists     676
About To Sleep          428
Need Attention          270
Promising               125
Can't Loose Them        108
New Customers            76
Name: count, dtype: int64


,Recency,Frequency,Monetary,R_Score,F_Score,M_Score,RFM_Score_Group,Segment
CustomerID,,,,,,,,
12346,326,34,77556.46,2,2,5,225,Hibernating
12347,2,253,5633.32,5,5,5,555,Champions
12348,75,51,2019.40,3,3,4,334,Need Attention
12349,19,175,4428.69,5,4,5,545,Champions
12350,310,17,334.40,2,2,2,222,Hibernating


In [28]:
# Save RFM results to a CSV file in my project folder
rfm.to_csv('Final_RFM_Results.csv')
print("File 'Final_RFM_Results.csv' saved successfully!")

File 'Final_RFM_Results.csv' saved successfully!


In [29]:
# Load the final result file back in to check it
final_df = pd.read_csv('Final_RFM_Results.csv')

# 1. Check Dimensions (Rows, Columns)
print("--- Dataset Dimensions ---")
print(f"Total Customers (Rows): {final_df.shape[0]}")
print(f"Total Attributes (Columns): {final_df.shape[1]}")

# 2. View the Column Names
print("\n--- Columns in Final Table ---")
print(final_df.columns.tolist())

# 3. Quick Summary of the Data
print("\n--- Data Summary ---")
print(final_df.info())

--- Dataset Dimensions ---
Total Customers (Rows): 5878
Total Attributes (Columns): 9

--- Columns in Final Table ---
['CustomerID', 'Recency', 'Frequency', 'Monetary', 'R_Score', 'F_Score', 'M_Score', 'RFM_Score_Group', 'Segment']

--- Data Summary ---
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5878 entries, 0 to 5877
Data columns (total 9 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   CustomerID       5878 non-null   int64  
 1   Recency          5878 non-null   int64  
 2   Frequency        5878 non-null   int64  
 3   Monetary         5878 non-null   float64
 4   R_Score          5878 non-null   int64  
 5   F_Score          5878 non-null   int64  
 6   M_Score          5878 non-null   int64  
 7   RFM_Score_Group  5878 non-null   int64  
 8   Segment          5878 non-null   object 
dtypes: float64(1), int64(7), object(1)
memory usage: 413.4+ KB
None


In [30]:
# Pulling the table
# Read using the engine
query = "SELECT * FROM Product_Summary"
Product_df = pd.read_sql(query, engine)

print(f"Data successfully pulled! Row count: {len(Product_df)}")
Product_df.head(5)



Data successfully pulled! Row count: 5285


,Product ID,Product Name,Category,Total Orders,Total Units Sold,Total Revenue
0,22418,10 COLOUR SPACEBOY PEN,Gifts & Stationery,472,12284,10211.30
1,22139,11 PC CERAMIC TEA SET POLKADOT,Kitchen & Dining,1,3,14.85
2,35962,12 ASS ZINC CHRISTMAS DECORATIONS,Home Decor & Furnishings,45,441,926.10
3,22436,12 COLOURED PARTY BALLOONS,Seasonal & Events,255,4227,2683.35
4,21448,12 DAISY PEGS IN WOOD BOX,Storage Baskets & Boxes,178,909,1499.85


In [31]:
print("--- Dataset Dimensions ---")

# Check dimensions (Rows, Columns)
print(f"Dataset Shape: {Product_df.shape}")

# Check total row count specifically
print(f"Total Rows: {len(Product_df)}")

# Check column names and data types
print(Product_df.info())

--- Dataset Dimensions ---
Dataset Shape: (5285, 6)
Total Rows: 5285
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5285 entries, 0 to 5284
Data columns (total 6 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Product ID        5285 non-null   object 
 1   Product Name      5285 non-null   object 
 2   Category          5285 non-null   object 
 3   Total Orders      5285 non-null   int64  
 4   Total Units Sold  5285 non-null   int64  
 5   Total Revenue     5285 non-null   float64
dtypes: float64(1), int64(2), object(3)
memory usage: 247.9+ KB
None
